## Create a file with word distributions and intruders

In [3]:
import pandas as pd
import numpy as np
import random

In [4]:
HOME_DIR = "Distributions-Results"
DATA_SET = "20NewsGroup"
MODEL = "prior_GMM"

In [5]:
import os
os.chdir(path="..")
cwd = os.getcwd()
cwd

'/home/dsi/ishonta/TM'

In [6]:
def get_top_words_for_topics(topic_word_df, top_n=8):
    top_words_by_topic = {}
    for topic_id in topic_word_df['Topic']:
        # Get the row corresponding to the current topic (excluding the 'Topic' column)
        topic_row = topic_word_df.loc[topic_word_df['Topic'] == topic_id].drop(columns='Topic').iloc[0]
        # Sort the row values in descending order and get the top N columns (words)
        top_words = topic_row.sort_values(ascending=False).head(top_n)
        
        # Collect the top N words (column names)
        top_words_by_topic[topic_id] = top_words.index.tolist()
    
    return top_words_by_topic

In [7]:
def get_GMM_top_words_for_topics(topic_word_df, top_n=8):
    top_words_by_topic = {}
    for topic_id in range(topic_word_df.shape[0]):
        # Get the row corresponding to the current topic (excluding the 'Topic' column)
        topic_row = topic_word_df.loc[topic_word_df['Topic'] == topic_id].drop(columns='Topic').iloc[0].dropna()
        # Sort the row values in descending order and get the top N columns (words)
        top_words = topic_row.head(top_n)
        
        # Collect the top N words (column names)
        top_words_by_topic[topic_id] = top_words.tolist()
    
    return top_words_by_topic

In [8]:
def get_intruder_words(topic_word_df, top_k=8, GMM=False, num__potential_intruder_words=10):
    """
    Identify intruder words for each topic.
    - Finds low-probability words in each topic.
    - Checks if these words are among the top-k words in another topic.
    - If a low-probability word is not found in any top-k list, a random high-probability word from another topic is selected.
    """
    # Get top-k words for all topics
    if not GMM:
        top_words_by_topic = get_top_words_for_topics(topic_word_df, top_n=top_k)
    else:
        top_words_by_topic = get_GMM_top_words_for_topics(topic_word_df, top_n=top_k)

    top_n = 30  # indicates the index from which we take the low probability words to be the intruder.
    intruder_words = {}

    for topic_id in topic_word_df['Topic'].unique():
        # Get the current topic row
        topic_row = topic_word_df.loc[topic_word_df['Topic'] == topic_id].drop(columns='Topic').iloc[0]
        topic_row = topic_row.dropna()

        # Find the lowest-probability words of that topic
        # Select words after the top `top_n` but not the lowest
        if not GMM:
            middle_prob_words = list(topic_row.sort_values()[top_n:top_n + num__potential_intruder_words].index)
        else:
            middle_prob_words = list(topic_row[top_n:top_n + num__potential_intruder_words])

        
        # shuffle in order to increase randomness in intruder words between different topics
        random.shuffle(middle_prob_words)

        intruder_word = None

        # Check if any low-probability word is an intruder
        for word in middle_prob_words:
            found_intruder = False

            # Check if the word appears in the top-k list of any other topic - if so, indicates word is probably meaning full
            for other_topic_id, top_k_words in top_words_by_topic.items():
                if other_topic_id != topic_id and word in top_k_words:
                    intruder_word = word
                    found_intruder = True
                    break  # Stop looking for other topics once an intruder is found

            # If the word is not found in any top-k list, try the next word in middle_prob_words. If found - break
            if found_intruder:
                break  

        # if no potential intruder was found to be in other topics in high probability
        if not found_intruder:
            # Get the list of words not in the current topic's top-k words - to avoid using intruders that have similar meaning with current topic.
            if not GMM:
                top_40_words_by_topic = get_top_words_for_topics(topic_word_df, top_n=40)
            else:
                top_40_words_by_topic = get_GMM_top_words_for_topics(topic_word_df, top_n=40)

            current_topic_top_40 = set(top_40_words_by_topic[topic_id])
            potential_intruders = set()
            for other_topic_id, top_k_words in top_words_by_topic.items():
                if other_topic_id != topic_id:
                    # Add words from the other topic that are not in the current topic's top-k
                    potential_intruders.update([word for word in top_k_words if word not in current_topic_top_40])

            # Randomly select one word from the potential intruders
            if potential_intruders:
                intruder_word = random.choice(list(potential_intruders))
            else:
                print(f"No suitable high-probability word found for topic {topic_id}")

        if intruder_word:
            intruder_words[topic_id] = intruder_word
        else:
            print(f"No intruder found for topic {topic_id}")

    return intruder_words


In [9]:
def insert_intruders_into_top_words(topic_word_df, top_n=8, GMM=False):
    """
    Insert intruder words into the top N words for each topic.
    """
    # Get the top words for each topic
    if not GMM:
        top_words_by_topic = get_top_words_for_topics(topic_word_df, top_n=top_n)
    else:
        top_words_by_topic = get_GMM_top_words_for_topics(topic_word_df, top_n=top_n)

    # Find the intruder words for each topic
    intruder_words = get_intruder_words(topic_word_df, top_k=8, GMM=GMM)

    # Insert the intruder word among the top words for each topic
    updated_top_words_by_topic = {}
    for topic_id, top_words in top_words_by_topic.items():
        intruder = intruder_words[topic_id]
        if intruder not in top_words:
            # Insert the intruder word (ensure the list remains unique)
            if GMM:
                top_words = top_words[1:]
            updated_top_words = top_words + [intruder]
            random.shuffle(updated_top_words)
            print(f"intruder: {intruder}")
            print(f"top words: {top_words}")
        else:
            print("GOT HERE")
            print(f"intruder: {intruder}")
            print(f"top words: {top_words}")
            updated_top_words = top_words
    
        updated_top_words_by_topic[topic_id] = updated_top_words

    return updated_top_words_by_topic, intruder_words

In [13]:
dir_path = f"helper/{DATA_SET}"

# Create directory if it doesn't exist
if not os.path.exists(dir_path):
    os.makedirs(dir_path)

for MODEL in ["prior_ScaSE"]:
    df = pd.read_csv(f"{HOME_DIR}/{DATA_SET}/{MODEL}_topic_word_distribution.csv")
    df = df.round(5)
    updated_top_words_by_topic, intruder_words = insert_intruders_into_top_words(df, top_n=8)
    updated_df = pd.DataFrame(updated_top_words_by_topic)
    intruders_df = pd.DataFrame(intruder_words, index=[0])
    updated_df.to_csv(f"{dir_path}/{MODEL}_intruder_check_.csv")
    intruders_df.to_csv(f"{dir_path}/{MODEL}_the_intruders_.csv")

In [10]:
HOME_DIR = "~/TM"
dir_path = HOME_DIR
MODEL = "GMM"

df = pd.read_csv(f"{HOME_DIR}/GMM_prior.csv", header=None)
df = df.round(5)
df.insert(0, "Topic", df.index)
topics = get_GMM_top_words_for_topics(df)
updated_top_words_by_topic, intruder_words = insert_intruders_into_top_words(df, top_n=10, GMM=True)
updated_df = pd.DataFrame(updated_top_words_by_topic)
intruders_df = pd.DataFrame(intruder_words, index=[0])
updated_df.to_csv(f"{dir_path}/{MODEL}_intruder_check.csv")
intruders_df.to_csv(f"{dir_path}/{MODEL}_the_intruders.csv")

SyntaxError: 'return' outside function (376983103.py, line 10)